# Part 4 — Cross-Section of Order Books, a Tick-Size Natural Experiment, 30-Second Depletion, and a Disclosure Test
This notebook runs the experiments that can still move both papers:

| Cell | Test | Paper | What it can show |
|---|---|---|---|
| X3 | T6 estimated on ~30 futures symbols, deep to thin | 2 | whether thinner books sit closer to (or across) the amplification threshold $C_3>0$ |
| X4 | depth mediation by liquidity tercile | 2 | whether a depth-carried overshoot appears in thin books |
| X5–X6 | **pre-registered** out-of-sample test in the thin tercile | 2 | whether $C_3$ forecasts next-hour adverse moves where books are thin |
| X7 | tick-size changes found in the data (difference-in-differences) | 2 | causal variation in refill $r$ |
| X8 | depletion at the ~30-second snapshot scale from every trade | 2 | whether the 1-minute "book refills the side being hit" result is an aggregation artefact |
| X9 | Binance top-trader vs all-account positions | 1 | what class-level disclosure buys: memory rates with vs without tagged data |
| X3 (Q columns) | feedback loop gain by symbol, with placebo | 1 | the feedback cross-section |

**Before running**
- Keep `USE_DRIVE = True` (cell X0). Everything is cached in Drive, so a disconnect only costs the current cell.
- The first run downloads about 30 symbols × 90 panel days (klines, depth, metrics), plus trade files for 5 symbols × 6 days. Expect 1–3 hours. `N_SYMBOLS = 20` makes it faster.
- **Run X5 before X6, then commit `prereg/P4_xsec.json` (for example to GitHub) before running X6.** X6 counts every time the 2025–2026 test period is opened; report that count in the paper.

**Rules:** do not change the X5 specification after running X6. If you must, disclose it. Whatever X6 shows goes in the paper.

In [ ]:
# ============================================================================
# CELL X0 — SETUP (self-contained; run first)
# Part 4: cross-section of order books, a tick-size natural experiment, 30-second
# depletion from trade data, and a disclosure test with Binance's class-level positions.
# Results go to the same results/ folder as Parts 1-2 when USE_DRIVE = True.
# ============================================================================
import os, sys, json, time, math, glob, re, io, zipfile, hashlib, subprocess, warnings, zlib
import urllib.request, urllib.error
from concurrent.futures import ThreadPoolExecutor, as_completed
import xml.etree.ElementTree as ET

IN_COLAB = "google.colab" in sys.modules
try:
    import pyarrow  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyarrow"], check=True)
import numpy as np
import pandas as pd
from scipy import stats, optimize
warnings.filterwarnings("ignore", category=RuntimeWarning)

USE_DRIVE = True
if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = "/content/drive/MyDrive/ceiling_suite"
elif IN_COLAB:
    BASE = "/content/ceiling_suite"
else:
    BASE = os.path.abspath("ceiling_suite_local")
RESULTS, PREREG = os.path.join(BASE, "results"), os.path.join(BASE, "prereg")
XDATA = os.path.join(BASE, "data_xsec")
for _d in (RESULTS, PREREG, XDATA):
    os.makedirs(_d, exist_ok=True)

SEED = 20261019
def rng(offset=0):
    return np.random.default_rng(SEED + offset)

def _clean(o):
    if isinstance(o, dict):
        return {str(k): _clean(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [_clean(v) for v in o]
    if isinstance(o, np.ndarray):
        return _clean(o.tolist())
    if isinstance(o, (np.floating, float)):
        return None if not np.isfinite(o) else float(o)
    if isinstance(o, np.integer):
        return int(o)
    if isinstance(o, np.bool_):
        return bool(o)
    return o

def record(test_id, passed, kind, metrics, criterion="", note=""):
    assert kind in {"theorem-check", "simulation", "real-data", "audit"}, kind
    status = "REPORT" if passed is None else ("PASS" if passed else "FAIL")
    out = {"test_id": test_id, "status": status, "kind": kind, "criterion": criterion, "note": note,
           "metrics": _clean(metrics), "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
           "seed": SEED}
    with open(os.path.join(RESULTS, f"{test_id}.json"), "w") as f:
        json.dump(out, f, indent=2)
    print(f"[{status}] {test_id} ({kind})" + (f" — {criterion}" if criterion else ""))
    if note:
        print("        note:", note)
    return out

def boot_ci(v, R, B=2000, stat=np.mean):
    v = np.asarray(v, float); v = v[np.isfinite(v)]
    if len(v) < 3:
        return [np.nan, np.nan]
    bs = [stat(v[R.integers(0, len(v), len(v))]) for _ in range(B)]
    return [float(np.quantile(bs, 0.025)), float(np.quantile(bs, 0.975))]

# ---- Sample design (fixed before looking at any Part 4 data) ----------------
N_SYMBOLS = 30                  # symbols in the cross-section, spread across liquidity (BTC and ETH always in)
PANEL_DAYS_OF_MONTH = (10, 24)  # same panel design as Part 1
PANEL_START, PANEL_END = "2023-01-01", "2026-09-30"
TRAIN_END, TEST_START = "2024-12-31", "2025-01-01"
LIQ_RANK_DATE = "2024-06-10"    # liquidity ranking day for choosing symbols (training period)
THREADS = 12
ROOT = "https://data.binance.vision/data/futures/um/daily"
print("BASE =", BASE)

def _http(url, method="GET", timeout=60, tries=3):
    last = None
    for i in range(tries):
        try:
            req = urllib.request.Request(url, method=method, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req, timeout=timeout) as r:
                return r.status, (r.read() if method == "GET" else b"")
        except urllib.error.HTTPError as e:
            if e.code in (403, 404):
                return e.code, b""
            last = e
        except Exception as e:
            last = e
        time.sleep(1.5 * (i + 1))
    raise RuntimeError(f"network error {url}: {last}")

def ds_url(ds, sym, date):
    if ds == "klines":
        return f"{ROOT}/klines/{sym}/1m/{sym}-1m-{date}.zip"
    return f"{ROOT}/{ds}/{sym}/{sym}-{ds}-{date}.zip"

def exists(ds, sym, date):
    return _http(ds_url(ds, sym, date), "HEAD", timeout=20)[0] == 200

DL_LOG = []
def fetch(ds, sym, date):
    """Zip bytes, checked against Binance's published .CHECKSUM when available; None if missing."""
    url = ds_url(ds, sym, date)
    code, raw = _http(url, timeout=180)
    if code != 200:
        return None
    sha = hashlib.sha256(raw).hexdigest()
    c2, chk = _http(url + ".CHECKSUM", timeout=30)
    verified = None
    if c2 == 200 and chk:
        if chk.decode().split()[0] != sha:
            raise RuntimeError(f"SHA-256 mismatch {url}")
        verified = True
    DL_LOG.append({"ds": ds, "symbol": sym, "date": date, "sha256": sha, "checksum_verified": verified})
    return raw

def read_zip_csv(raw, default_cols=None):
    with zipfile.ZipFile(io.BytesIO(raw)) as z:
        name = [n for n in z.namelist() if n.endswith(".csv")][0]
        txt = z.read(name).decode("utf-8", errors="ignore")
    first = txt.split("\n", 1)[0]
    has_header = bool(re.search(r"[A-Za-z_]", first.split(",")[0]))
    df = pd.read_csv(io.StringIO(txt), header=0 if has_header else None)
    if not has_header and default_cols is not None:
        df.columns = default_cols[:df.shape[1]]
    df.columns = [str(c).strip().lower() for c in df.columns]
    return df

def to_utc_seconds(col):
    s = pd.Series(col)
    if pd.api.types.is_numeric_dtype(s):
        v = s.astype("int64").to_numpy()
        return np.where(v > 1e14, v // 1_000_000, np.where(v > 1e11, v // 1000, v))
    ts = pd.to_datetime(s, utc=True)
    return ((ts - pd.Timestamp("1970-01-01", tz="UTC")) // pd.Timedelta("1s")).astype("int64").to_numpy()

def panel_dates(start=PANEL_START, end=PANEL_END):
    ds = pd.date_range(start, end, freq="D")
    return [d.strftime("%Y-%m-%d") for d in ds if d.day in PANEL_DAYS_OF_MONTH]


In [ ]:
# ============================================================================
# CELL X1 — CHOOSE THE SYMBOLS (before any analysis)
# 1. List USD-M perpetuals that have bookDepth on data.binance.vision (bucket listing;
#    falls back to a fixed list of long-lived contracts if the listing is unavailable).
# 2. Keep symbols with depth both early (2023) and late (2026), so every symbol spans
#    the training and test periods.
# 3. Rank by quote volume on LIQ_RANK_DATE (a training day) and take N_SYMBOLS spread
#    evenly across the ranking, so the sample runs from the deepest books to thin ones.
# ============================================================================
FALLBACK = ("BTCUSDT ETHUSDT BNBUSDT XRPUSDT ADAUSDT DOGEUSDT SOLUSDT DOTUSDT LTCUSDT LINKUSDT BCHUSDT TRXUSDT "
            "AVAXUSDT ATOMUSDT ETCUSDT FILUSDT NEARUSDT AAVEUSDT UNIUSDT SANDUSDT MANAUSDT AXSUSDT XLMUSDT ALGOUSDT "
            "VETUSDT EOSUSDT XTZUSDT THETAUSDT ICPUSDT EGLDUSDT CHZUSDT ENJUSDT ZILUSDT KAVAUSDT RUNEUSDT GALAUSDT "
            "CRVUSDT DYDXUSDT 1INCHUSDT COMPUSDT SNXUSDT ZECUSDT DASHUSDT NEOUSDT IOTAUSDT ONTUSDT QTUMUSDT BATUSDT "
            "ZRXUSDT KNCUSDT").split()

def list_bucket_symbols(ds="bookDepth"):
    url0 = ("https://s3-ap-northeast-1.amazonaws.com/data.binance.vision?delimiter=/"
            f"&prefix=data/futures/um/daily/{ds}/")
    syms, marker = [], ""
    for _ in range(20):
        code, raw = _http(url0 + (f"&marker={marker}" if marker else ""), timeout=60)
        if code != 200 or not raw:
            return []
        root = ET.fromstring(raw)
        ns = {"s": root.tag.split("}")[0].strip("{")} if root.tag.startswith("{") else {}
        pref = [e.text for e in (root.findall(".//s:CommonPrefixes/s:Prefix", ns) if ns else root.findall(".//CommonPrefixes/Prefix"))]
        syms += [p.rstrip("/").split("/")[-1] for p in pref]
        trunc = (root.find("s:IsTruncated", ns) if ns else root.find("IsTruncated"))
        if trunc is None or trunc.text != "true" or not pref:
            break
        marker = pref[-1]
    return sorted(set(syms))

try:
    listed = list_bucket_symbols()
except Exception as e:
    print("bucket listing failed:", e); listed = []
cands = [s for s in listed if re.fullmatch(r"[A-Z0-9]+USDT", s)] or FALLBACK
print(f"candidates: {len(cands)} ({'bucket listing' if listed else 'fallback list'})")

EARLY, LATE = "2023-03-10", "2026-06-10"
def span_ok(s):
    return exists("bookDepth", s, EARLY) and exists("bookDepth", s, LATE)
with ThreadPoolExecutor(THREADS) as ex:
    keep = [s for s, ok in zip(cands, ex.map(span_ok, cands)) if ok]
print(f"with depth in both {EARLY} and {LATE}: {len(keep)}")

KL_COLS = ["open_time", "open", "high", "low", "close", "volume", "close_time", "quote_volume",
           "count", "taker_buy_volume", "taker_buy_quote_volume", "ignore"]
def day_quote_volume(s):
    raw = fetch("klines", s, LIQ_RANK_DATE)
    return np.nan if raw is None else float(read_zip_csv(raw, KL_COLS)["quote_volume"].astype(float).sum())
with ThreadPoolExecutor(THREADS) as ex:
    qv = dict(zip(keep, ex.map(day_quote_volume, keep)))
rank = sorted([s for s in keep if np.isfinite(qv[s])], key=lambda s: -qv[s])
must = [s for s in ("BTCUSDT", "ETHUSDT") if s in rank]
rest = [s for s in rank if s not in must]
k = max(N_SYMBOLS - len(must), 0)
pick = [rest[int(round(i))] for i in np.linspace(0, len(rest) - 1, min(k, len(rest)))] if rest else []
SYMBOLS = must + [s for s in pick if s not in must]
print("symbols:", SYMBOLS)
record("X1_universe", None, "audit",
       {"source": "bucket listing" if listed else "fallback list", "n_candidates": len(cands),
        "n_span_ok": len(keep), "symbols": SYMBOLS,
        "quote_volume_musd_on_rank_date": {s: qv[s] / 1e6 for s in SYMBOLS}, "rank_date": LIQ_RANK_DATE},
       note="Symbols are fixed here, from a training-period ranking, before any T6 statistic is computed.")


In [ ]:
# ============================================================================
# CELL X2 — DOWNLOAD, VALIDATE AND BUILD 1-MINUTE PANELS FOR EVERY SYMBOL
# Same depth validation rules as Part 2 (V1-V5), applied to every symbol:
#   V1 bands -5,-2,-1,+1,+2,+5 present; V2 depth monotone in distance; V3 implied average price
#   at ±1% within 3% of the 1-min close; V4 no runs of >10 identical snapshots; V5 >= 80% of minutes valid.
# Also stored per day: the inferred price tick (for X7) and Binance's class-level position
# ratios from `metrics` (for X9).
# ============================================================================
NEED_BANDS = [-5, -2, -1, 1, 2, 5]

def nice_tick(x):
    """Snap a positive increment to the nearest 1, 2 or 5 x 10^k (exchange tick grids)."""
    if not np.isfinite(x) or x <= 0:
        return np.nan
    e = math.floor(math.log10(x)); m = x / 10 ** e
    c = min([1, 2, 5, 10], key=lambda v: abs(math.log(m / v)))
    return c * 10 ** e

def infer_tick(k):
    v = np.unique(np.round(k[["open", "high", "low", "close"]].astype(float).to_numpy().ravel(), 10))
    dv = np.diff(v); dv = dv[dv > 0]
    return nice_tick(np.quantile(dv, 0.02)) if len(dv) > 5 else np.nan

def validated_snapshots(sym, date, close_by_minute, t0):
    """Validated bookDepth snapshots: DataFrame indexed by second with bid/ask notional at 1,2,5% ($M)."""
    raw = fetch("bookDepth", sym, date)
    if raw is None:
        return None, {"snapshots": 0}
    b = read_zip_csv(raw)
    b["sec"] = to_utc_seconds(b["timestamp"])
    b["percentage"] = pd.to_numeric(b["percentage"], errors="coerce").round().astype("Int64")
    piv_n = b.pivot_table(index="sec", columns="percentage", values="notional", aggfunc="last")
    piv_d = b.pivot_table(index="sec", columns="percentage", values="depth", aggfunc="last")
    n0 = len(piv_n)
    ok = piv_n.reindex(columns=NEED_BANDS).notna().all(axis=1) & piv_d.reindex(columns=NEED_BANDS).notna().all(axis=1)
    piv_n, piv_d = piv_n[ok], piv_d[ok]
    mono = (piv_d[-2] >= piv_d[-1]) & (piv_d[-5] >= piv_d[-2]) & (piv_d[2] >= piv_d[1]) & (piv_d[5] >= piv_d[2])
    minute = ((piv_n.index.to_numpy() - t0) // 60).clip(0, 1439)
    ref = close_by_minute[minute]
    imp_b, imp_a = piv_n[-1] / piv_d[-1], piv_n[1] / piv_d[1]
    price_ok = (np.abs(imp_b / ref - 1) < 0.03) & (np.abs(imp_a / ref - 1) < 0.03)
    sig = piv_n[NEED_BANDS].round(2).astype(str).agg("|".join, axis=1)
    runlen = sig.groupby((sig != sig.shift()).cumsum()).transform("size")
    not_stale = runlen <= 10
    good = mono & price_ok & not_stale
    g = piv_n[good]
    snap = pd.DataFrame({"bid1": g[-1], "ask1": g[1], "bid2": g[-2], "ask2": g[2], "bid5": g[-5], "ask5": g[5]}) / 1e6
    st = {"snapshots": n0, "drop_V1": int((~ok).sum()), "drop_V2": int((~mono).sum()),
          "drop_V3": int((~price_ok).sum()), "drop_V4": int((~not_stale).sum())}
    return snap, st

def build_day(sym, date):
    t0 = int(pd.Timestamp(f"{date} 00:00:00", tz="UTC").timestamp())
    idx = np.arange(1440)
    raw = fetch("klines", sym, date)
    if raw is None:
        return None, {"symbol": sym, "date": date, "reason": "no klines"}
    k = read_zip_csv(raw, KL_COLS)
    tick = infer_tick(k)
    k = k.assign(m=(to_utc_seconds(k["open_time"]) - t0) // 60).drop_duplicates("m").set_index("m").reindex(idx)
    out = pd.DataFrame(index=idx)
    out["close"] = k["close"].astype(float).ffill().bfill()
    out["flow"] = (2 * k["taker_buy_quote_volume"].astype(float) - k["quote_volume"].astype(float)).fillna(0) / 1e6
    out["vol"] = k["quote_volume"].astype(float).fillna(0) / 1e6
    st = {"symbol": sym, "date": date, "tick": tick}
    snap, s2 = validated_snapshots(sym, date, out["close"].to_numpy(), t0)
    st.update(s2)
    for c in ["bid1", "ask1", "bid2", "ask2", "bid5", "ask5"]:
        out[c] = np.nan
    if snap is not None and len(snap):
        mm = ((snap.index.to_numpy() - t0) // 60).clip(0, 1439)
        sm = snap.assign(m=mm).groupby("m").last().reindex(idx).ffill(limit=2)
        for c in snap.columns:
            out[c] = sm[c].to_numpy()
    st["minute_coverage"] = float(out["bid1"].notna().mean())
    st["depth_day_ok"] = st["minute_coverage"] >= 0.80
    for c in ["oi", "top_pos_ratio", "top_acc_ratio", "acc_ratio", "taker_ratio"]:
        out[c] = np.nan
    raw = fetch("metrics", sym, date)
    if raw is not None:
        mt = read_zip_csv(raw)
        tcol = "create_time" if "create_time" in mt.columns else mt.columns[0]
        mm = ((to_utc_seconds(mt[tcol]) - t0) // 60).clip(0, 1439)
        for c, src, scale in [("oi", "sum_open_interest_value", 1e6),
                              ("top_pos_ratio", "sum_toptrader_long_short_ratio", 1.0),
                              ("top_acc_ratio", "count_toptrader_long_short_ratio", 1.0),
                              ("acc_ratio", "count_long_short_ratio", 1.0),
                              ("taker_ratio", "sum_taker_long_short_vol_ratio", 1.0)]:
            if src in mt.columns:
                s = pd.Series(pd.to_numeric(mt[src], errors="coerce").to_numpy() / scale, index=mm)
                out[c] = s.groupby(level=0).last().reindex(idx).to_numpy()   # 5-min values, not filled
    out["symbol"], out["date"] = sym, date
    return out, st

MIN_DIR = os.path.join(XDATA, "minute")
def get_day(sym, date):
    d_ = os.path.join(MIN_DIR, sym); os.makedirs(d_, exist_ok=True)
    pq, js = os.path.join(d_, f"{date}.parquet"), os.path.join(d_, f"{date}.json")
    if os.path.exists(js):
        st = json.load(open(js))
        return (pd.read_parquet(pq) if os.path.exists(pq) else None), st
    df, st = build_day(sym, date)
    if df is not None:
        df.to_parquet(pq)
    json.dump(_clean(st), open(js, "w"))
    return df, st

PDATES = panel_dates()
jobs = [(s, d) for s in SYMBOLS for d in PDATES]
print(f"building {len(jobs)} symbol-days ({len(SYMBOLS)} symbols x {len(PDATES)} panel days); cached ones are skipped")
PANEL, VST = {}, []
with ThreadPoolExecutor(THREADS) as ex:
    futs = {ex.submit(get_day, s, d): (s, d) for s, d in jobs}
    for i, f in enumerate(as_completed(futs), 1):
        s, d = futs[f]
        try:
            df, st = f.result(); VST.append(st)
            if df is not None:
                PANEL[(s, d)] = df
        except Exception as e:
            VST.append({"symbol": s, "date": d, "reason": f"error: {e}"})
        if i % 200 == 0:
            print(f"   {i}/{len(jobs)}")
vst = pd.DataFrame(VST)
vst["depth_day_ok"] = vst.get("depth_day_ok", pd.Series(False, index=vst.index)).fillna(False).astype(bool)
OK = sorted([(r.symbol, r.date) for r in vst.itertuples() if r.depth_day_ok and (r.symbol, r.date) in PANEL])
pd.DataFrame(DL_LOG).to_csv(os.path.join(RESULTS, "xsec_manifest_sha256.csv"), index=False)
by_sym = vst.groupby("symbol")["depth_day_ok"].agg(["sum", "count"]).rename(columns={"sum": "valid", "count": "requested"})
print(by_sym.to_string())
n_err = int(vst.get("reason", pd.Series(dtype=str)).astype(str).str.startswith("error").sum()) if "reason" in vst else 0
record("X2_build", None, "audit",
       {"symbol_days_requested": len(jobs), "symbol_days_valid": len(OK), "n_error": n_err,
        "by_symbol": by_sym.to_dict("index"),
        "snapshot_drops": {c: int(vst[c].fillna(0).sum()) for c in ["snapshots", "drop_V1", "drop_V2", "drop_V3", "drop_V4"] if c in vst}},
       note="Report the valid counts by symbol in the data appendix.")


## Training-period analysis (≤ 2024-12-31)

In [ ]:
# ============================================================================
# CELL X3 — T6 ACROSS THE CROSS-SECTION (training days only, <= 2024-12-31)
# Same day-by-day estimator as Part 2 (D3), on every symbol:
#   price:    Δp_t = κ u_t + φ d_{t-1} - (1-a) x_{t-1} + e
#   depth:    d_t  = (1-r) d_{t-1} + ε u_t + e
#   momentum: u_t  = (1-η) u_{t-1} + ηq x_{t-1} + e
#   C3 = φε + κ²ηq - (1-a)κ  (amplification iff C3 > 0 in the 2-state case with η q = 0)
# Flow u is scaled by the symbol's median 1-minute |flow| so κ, φ, ε are comparable across symbols
# (C3's sign does not depend on this scaling).
# Also: the feedback loop gain Q of Paper 1 (VAR on x and next-minute flow) and its placebo.
# Liquidity = median depth within 1% of mid (bid1 + ask1, $M) on training days.
# ============================================================================
def prep(df, flow_scale):
    p = np.log(df["close"].to_numpy()) * 1e4
    u = df["flow"].to_numpy() / flow_scale
    tot = (df["bid1"] + df["ask1"]).to_numpy()
    d = ((df["bid1"] - df["ask1"]) / (df["bid1"] + df["ask1"])).to_numpy()
    x = p - pd.Series(p).ewm(span=60, adjust=False).mean().to_numpy()
    return p, u, d, x, tot

def ols(y, X):
    ok = np.all(np.isfinite(X), 1) & np.isfinite(y)
    if ok.sum() < X.shape[1] + 20:
        return None
    return np.linalg.lstsq(np.column_stack([np.ones(ok.sum()), X[ok]]), y[ok], rcond=None)[0][1:]

def impulse_t6(a, phi, kappa, r, eps, eta, q, L=120):
    x = np.zeros(L); dd = np.zeros(L); f = np.zeros(L)
    for t in range(L):
        if t > 0:
            f[t] = (1 - eta) * f[t - 1] + eta * q * x[t - 1]
        tot = (1.0 if t == 0 else 0.0) + f[t]
        x[t] = (a * x[t - 1] + phi * dd[t - 1] if t else 0) + kappa * tot
        dd[t] = ((1 - r) * dd[t - 1] if t else 0) + eps * tot
        if abs(x[t]) > 1e9:
            return np.full(L, np.nan)
    return x

def fit_t6(p, u, d, x):
    dp = np.diff(p, prepend=np.nan)
    bp = ols(dp[1:], np.column_stack([u[1:], d[:-1], x[:-1]]))
    bd = ols(d[1:], np.column_stack([d[:-1], u[1:]]))
    bm = ols(u[1:], np.column_stack([u[:-1], x[:-1]]))
    if bp is None or bd is None or bm is None:
        return None
    kappa, phi, a = bp[0], bp[1], 1 + bp[2]
    r, eps, eta = 1 - bd[0], bd[1], 1 - bm[0]
    q = bm[1] / eta if abs(eta) > 1e-6 else np.nan
    h = impulse_t6(a, phi, kappa, r, eps, eta, q)
    ok = np.all(np.isfinite(h)) and abs(h[0]) > 1e-12
    return {"kappa": kappa, "phi": phi, "a": a, "r": r, "eps": eps, "eta": eta, "q": q,
            "C2": phi * eps - (1 - a) * kappa, "C3": phi * eps + kappa ** 2 * eta * q - (1 - a) * kappa,
            "peak_ratio": float(np.max(h) / h[0]) if ok else np.nan, "peak_minute": int(np.argmax(h)) if ok else -1}

def loop_gain(p, u, span=30):
    x = p - pd.Series(p).ewm(span=span, adjust=False).mean().to_numpy()
    f = np.roll(u, -1)[:-1]; x = x[:-1]
    Z = np.column_stack([x, f])
    Y, X = Z[1:], np.column_stack([np.ones(len(Z) - 1), Z[:-1]])
    ok = np.all(np.isfinite(Y), 1) & np.all(np.isfinite(X), 1)
    if ok.sum() < 100:
        return np.nan
    J = np.linalg.lstsq(X[ok], Y[ok], rcond=None)[0][1:].T
    return float(J[0, 1] * J[1, 0] / (1 - J[1, 1])) if abs(1 - J[1, 1]) > 1e-9 else np.nan

FLOW_SCALE = {}
for s in SYMBOLS:
    v = [np.nanmedian(np.abs(PANEL[(s2, d)]["flow"].to_numpy())) for (s2, d) in OK if s2 == s and d <= TRAIN_END]
    FLOW_SCALE[s] = float(np.nanmedian(v)) if v and np.nanmedian(v) > 0 else 1.0

Rq = rng(31)
def symday_row(s, d):
    df = PANEL[(s, d)]
    p, u, dd_, x, tot = prep(df, FLOW_SCALE[s])
    est = fit_t6(p, u, dd_, x)
    if est is None:
        return None
    rv = float(np.nanstd(np.diff(p)))
    pl = np.cumsum(np.random.default_rng(zlib.crc32(f"{s}{d}".encode())).normal(0, rv, len(p)))   # placebo walk
    est.update({"symbol": s, "date": d, "rv": rv, "depth1_musd": float(np.nanmedian(tot)),
                "Q": loop_gain(p, u), "Q_placebo": loop_gain(pl, u)})
    return est

rows = [r for r in (symday_row(s, d) for (s, d) in OK) if r]
XS = pd.DataFrame(rows)
XS["split"] = np.where(XS["date"] <= TRAIN_END, "train", "test")
XS.to_parquet(os.path.join(XDATA, "symbol_days.parquet"))
tr = XS[XS["split"] == "train"]
sym = tr.groupby("symbol").agg(depth1=("depth1_musd", "median"), C3_mean=("C3", "mean"), C3_median=("C3", "median"),
                               share_C3_pos=("C3", lambda v: float((v > 0).mean())), share_C2_pos=("C2", lambda v: float((v > 0).mean())),
                               phi=("phi", "median"), eps=("eps", "median"), r=("r", "median"),
                               peak_ratio=("peak_ratio", "median"), share_peak_gt1=("peak_ratio", lambda v: float((v > 1.0001).mean())),
                               Q=("Q", "mean"), Q_placebo=("Q_placebo", "mean"), n_days=("C3", "size"))
sym = sym[sym["n_days"] >= 10].sort_values("depth1", ascending=False)
# liquidity terciles fixed on TRAINING depth (used unchanged in X5/X6)
sym["tercile"] = pd.qcut(sym["depth1"].rank(method="first"), 3, labels=["thin", "mid", "thick"]).astype(str)
TERCILE = sym["tercile"].to_dict()
print(sym.round(4).to_string())

Rb = rng(32); S = sym.reset_index()
def spearman_boot(col):
    rho0 = stats.spearmanr(np.log(S["depth1"]), S[col]).statistic
    bs = [stats.spearmanr(np.log(S["depth1"].iloc[i]), S[col].iloc[i]).statistic
          for i in (Rb.integers(0, len(S), len(S)) for _ in range(2000))]
    return {"rho": float(rho0), "ci": [float(np.nanquantile(bs, .025)), float(np.nanquantile(bs, .975))]}
terc = {t: {"n_symbols": int((sym["tercile"] == t).sum()),
            "share_C3_pos_pooled": float((tr[tr["symbol"].map(TERCILE) == t]["C3"] > 0).mean()),
            "mean_C3_pooled": float(tr[tr["symbol"].map(TERCILE) == t]["C3"].mean()),
            "mean_C3_ci": boot_ci(tr[tr["symbol"].map(TERCILE) == t]["C3"], Rb),
            "median_phi": float(tr[tr["symbol"].map(TERCILE) == t]["phi"].median()),
            "median_eps": float(tr[tr["symbol"].map(TERCILE) == t]["eps"].median()),
            "share_eps_pos": float((tr[tr["symbol"].map(TERCILE) == t]["eps"] > 0).mean()),
            "mean_Q": float(tr[tr["symbol"].map(TERCILE) == t]["Q"].mean()),
            "mean_Q_ci": boot_ci(tr[tr["symbol"].map(TERCILE) == t]["Q"], Rb)}
        for t in ["thick", "mid", "thin"]}
print(pd.DataFrame(terc).T.to_string())
record("X3_T6_cross_section", None, "real-data",
       {"by_symbol": sym.to_dict("index"), "by_tercile": terc,
        "spearman_logdepth_vs_C3": spearman_boot("C3_mean"),
        "spearman_logdepth_vs_shareC3pos": spearman_boot("share_C3_pos"),
        "spearman_logdepth_vs_Q": spearman_boot("Q"),
        "n_symbol_days_train": len(tr), "flow_scale_musd": FLOW_SCALE},
       note="T6 predicts higher C3 in thinner books: negative Spearman with log depth, CI below 0. "
            "Q vs placebo per symbol is Paper 1's feedback cross-section.")


In [ ]:
# ============================================================================
# CELL X4 — IS THERE A DEPTH-CARRIED OVERSHOOT IN THIN BOOKS? (training days)
# Local projections per symbol-day, as in Part 2 (D4): y_k = p_{t+k} - p_{t-1} on u_t
# (flow in units of the symbol's typical |flow|), 5 lags of Δp, u, d as controls:
#   (A) total; (B) + cumulative future flow; (C) + future flow and cumulative depth imbalance.
# Averaged by liquidity tercile. Overshoot = max_{k>=1} β_k - β_0, also relative to β_0 (unit-free).
# T6 predicts: overshoot > 0 in (B) in thin books, smaller in (C).
# ============================================================================
H_MIN = [0, 1, 2, 3, 5, 10, 15, 30]
def lp_day(df, flow_scale, nl=5):
    p, u, dd_, x, tot = prep(df, flow_scale)
    dp = np.diff(p, prepend=p[0]); n = len(p); t = np.arange(nl + 1, n - max(H_MIN) - 1)
    if not np.isfinite(dd_[t]).mean() > 0.9:
        return None
    L = np.column_stack([dp[t - j] for j in range(1, nl + 1)] + [u[t - j] for j in range(1, nl + 1)] +
                        [dd_[t - j] for j in range(1, nl + 1)])
    cu = np.concatenate([[0.0], np.cumsum(u)]); cd = np.concatenate([[0.0], np.cumsum(np.nan_to_num(dd_))])
    out = {"A": [], "B": [], "C": []}
    for k in H_MIN:
        y = p[t + k] - p[t - 1]
        XA = np.column_stack([u[t], L])
        XB = np.column_stack([XA, cu[t + k + 1] - cu[t + 1]]) if k else XA
        XC = np.column_stack([XB, cd[t + k] - cd[t]]) if k else XB
        for key, X in (("A", XA), ("B", XB), ("C", XC)):
            b = ols(y, X); out[key].append(b[0] if b is not None else np.nan)
    return {k: np.array(v) for k, v in out.items()}

LP = {}
for (s, d) in OK:
    if d <= TRAIN_END and s in TERCILE:
        r_ = lp_day(PANEL[(s, d)], FLOW_SCALE[s])
        if r_ is not None:
            LP[(s, d)] = r_
R4 = rng(44); res4 = {}
for terc in ["thick", "mid", "thin"]:
    keys = [k for k in LP if TERCILE[k[0]] == terc]
    res4[terc] = {"n_symbol_days": len(keys)}
    for v in ("A", "B", "C"):
        M = np.array([LP[k][v] for k in keys]); M = M[np.all(np.isfinite(M), 1)] if len(M) else M
        if len(M) < 20:
            continue
        mean = M.mean(0)
        boots = np.array([M[R4.integers(0, len(M), len(M))].mean(0) for _ in range(1000)])
        over = boots[:, 1:].max(1) - boots[:, 0]; rel = over / boots[:, 0]
        res4[terc][v] = {"beta": mean, "overshoot": float(mean[1:].max() - mean[0]),
                         "overshoot_ci": [float(np.quantile(over, .025)), float(np.quantile(over, .975))],
                         "overshoot_rel": float((mean[1:].max() - mean[0]) / mean[0]),
                         "overshoot_rel_ci": [float(np.quantile(rel, .025)), float(np.quantile(rel, .975))],
                         "peak_minute": H_MIN[int(np.argmax(mean))]}
        print(f"{terc:5s} ({v}) β {np.round(mean, 3)} overshoot/β0 {res4[terc][v]['overshoot_rel']:.3f} "
              f"CI {np.round(res4[terc][v]['overshoot_rel_ci'], 3)}")
record("X4_mediation_by_liquidity", None, "real-data", {"horizons_min": H_MIN, "by_tercile": res4},
       note="Support for T6 in thin books = (B) overshoot CI above 0 in the thin tercile, smaller in (C).")


## Pre-register, then test once on 2025–2026

In [ ]:
# ============================================================================
# CELL X5 — HOURLY BLOCKS FOR EVERY SYMBOL + PRE-REGISTRATION (run before X6)
# Blocks: hours 6..23 UTC; features from minutes [t-360, t), outcomes from (t, t+60].
# The spec below is hashed and saved BEFORE the 2025-2026 blocks are evaluated in X6.
# ============================================================================
def day_blocks(s, d):
    df = PANEL[(s, d)]
    p, u, dd_, x, tot = prep(df, FLOW_SCALE[s])
    oi = df["oi"].ffill().to_numpy(); vol = df["vol"].to_numpy(); rows = []
    for hr in range(6, 24):
        t = hr * 60
        if t + 1 >= 1440:
            continue
        w, h, f = slice(t - 360, t), slice(t - 60, t), slice(t, min(t + 61, 1440))
        est = fit_t6(p[w], u[w], dd_[w], x[w])
        r60, r360 = np.diff(p[h]), np.diff(p[w])
        rows.append({"symbol": s, "date": d, "hour": hr,
                     "C3_hat": est["C3"] if est else np.nan, "peak_ratio_hat": est["peak_ratio"] if est else np.nan,
                     "rv_60": np.nanstd(r60), "rv_360": np.nanstd(r360), "abs_ret_60": abs(p[t - 1] - p[t - 60]),
                     "flow_imb_60": np.nansum(u[h]) / (np.nansum(np.abs(u[h])) + 1e-12),
                     "log_vol_60": np.log1p(np.nansum(vol[h])),
                     "log_depth1": np.log(tot[t - 1]) if np.isfinite(tot[t - 1]) and tot[t - 1] > 0 else np.nan,
                     "imbalance": dd_[t - 1],
                     "oi_chg_60": (oi[t - 1] / oi[t - 60] - 1) * 100 if np.isfinite(oi[t - 60]) and oi[t - 60] > 0 else np.nan,
                     "MAE_60": float(np.nanmax(np.abs(p[f] - p[t - 1]))),
                     "OI_drop_60": float((oi[t - 1] - np.nanmin(oi[f])) / oi[t - 1] * 100) if np.isfinite(oi[t - 1]) and oi[t - 1] > 0 else np.nan,
                     "depth_collapse_60": float((1 - np.nanmin(tot[f]) / tot[t - 1]) * 100) if np.isfinite(tot[t - 1]) and tot[t - 1] > 0 else np.nan})
    return rows

BLK = os.path.join(XDATA, f"blocks_{len(SYMBOLS)}sym.parquet")
want = {(s, d) for (s, d) in OK if s in TERCILE}
if os.path.exists(BLK) and set(map(tuple, pd.read_parquet(BLK)[["symbol", "date"]].drop_duplicates().to_numpy())) >= want:
    XB = pd.read_parquet(BLK)
else:
    rows = []
    for i, (s, d) in enumerate(sorted(want)):
        rows += day_blocks(s, d)
        if (i + 1) % 300 == 0:
            print(f"   blocks: {i+1}/{len(want)} symbol-days")
    XB = pd.DataFrame(rows); XB.to_parquet(BLK)
XB["tercile"] = XB["symbol"].map(TERCILE)
XB["split"] = np.where(XB["date"] <= TRAIN_END, "train", np.where(XB["date"] >= TEST_START, "test", "gap"))
print(XB.groupby(["tercile", "split"]).size().unstack())

# leakage audit: scramble everything from t on; features must not change
Rl = rng(55); fails = 0; keys = sorted(want)
for _ in range(30):
    s, d = keys[int(Rl.integers(len(keys)))]; hr = int(Rl.integers(6, 23)); t = hr * 60
    a_ = [r for r in day_blocks(s, d) if r["hour"] == hr][0]
    saved = PANEL[(s, d)]; df2 = saved.copy()
    for c in ["close", "flow", "vol", "bid1", "ask1", "oi"]:
        v = df2[c].to_numpy().copy(); v[t:] = v[t:][::-1] * 1.7; df2[c] = v
    PANEL[(s, d)] = df2
    b_ = [r for r in day_blocks(s, d) if r["hour"] == hr][0]
    PANEL[(s, d)] = saved
    feats = [k for k in a_ if k not in ("symbol", "date", "hour", "MAE_60", "OI_drop_60", "depth_collapse_60")]
    fails += any(not np.isclose(a_[k], b_[k], equal_nan=True) for k in feats)
record("X5_no_lookahead", fails == 0, "audit", {"trials": 30, "failures": fails},
       criterion="hourly features unchanged when all data from t onward are scrambled")

PREREG_X = {
    "question": "In the thinnest third of Binance USD-M order books, does the T6 amplification statistic C3_hat "
                "(estimated from the previous 6 hours) predict the next hour's maximum adverse excursion beyond controls?",
    "data": "Binance USD-M futures, symbols fixed in cell X1; panel days 10th and 24th of each month; validation V1-V5",
    "terciles": "liquidity terciles from median training-period depth within 1% of mid (cell X3), fixed",
    "unit": "symbol-hour blocks, hours 6-23 UTC; features [t-360, t); outcomes (t, t+60]",
    "controls": ["rv_60", "rv_360", "abs_ret_60", "flow_imb_60", "log_vol_60", "log_depth1", "imbalance", "oi_chg_60", "symbol fixed effects"],
    "primary": "tercile=thin, outcome=MAE_60, candidate=C3_hat winsorized at the thin tercile's train 1%/99%",
    "secondary": ["thin: MAE_60 | peak_ratio_hat", "thin: depth_collapse_60 | C3_hat", "thin: OI_drop_60 | C3_hat",
                  "mid: MAE_60 | C3_hat", "thick: MAE_60 | C3_hat", "all symbols: MAE_60 | C3_hat"],
    "train": "dates <= 2024-12-31", "test": "dates >= 2025-01-01",
    "model": "OLS; tail = outcome above train 90th percentile; score = fitted value",
    "inference": "Clark-West (Newey-West 6 lags); bootstrap 2,000 resampling dates (all symbols on a date together) for ΔR², ΔAUC",
    "success": "primary: Clark-West one-sided p < 0.05 AND bootstrap 95% CI of ΔR² excludes 0; secondary Holm-adjusted",
}
spec_txt = json.dumps(PREREG_X, indent=2, sort_keys=True)
X_HASH = hashlib.sha256(spec_txt.encode()).hexdigest()
px_path = os.path.join(PREREG, "P4_xsec.json")
if not os.path.exists(px_path):
    open(px_path, "w").write(spec_txt); print("pre-registration written:", px_path)
elif hashlib.sha256(open(px_path).read().encode()).hexdigest() != X_HASH:
    print("!! WARNING: spec differs from the registered file. Any change after seeing results must be disclosed.")
print("spec SHA-256:", X_HASH, "\nCommit prereg/P4_xsec.json (e.g. to GitHub) before running X6.")


In [ ]:
# ============================================================================
# CELL X6 — HELD-OUT TEST, 2025-2026 (every evaluation is counted)
# ============================================================================
CTRL = ["rv_60", "rv_360", "abs_ret_60", "flow_imb_60", "log_vol_60", "log_depth1", "imbalance", "oi_chg_60"]

def unlock_x():
    pth = os.path.join(PREREG, "test_set_unlocks_xsec.json")
    n = json.load(open(pth))["n"] + 1 if os.path.exists(pth) else 1
    json.dump({"n": n, "utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()), "spec_sha256": X_HASH}, open(pth, "w"))
    return n

def auc_(y, s):
    pos, neg = s[y == 1], s[y == 0]
    if len(pos) == 0 or len(neg) == 0:
        return np.nan
    rk = stats.rankdata(np.concatenate([pos, neg]))
    return (rk[:len(pos)].sum() - len(pos) * (len(pos) + 1) / 2) / (len(pos) * len(neg))

def cw_test(y, fb, ff, lags=6):
    adj = (y - fb) ** 2 - ((y - ff) ** 2 - (fb - ff) ** 2)
    m = adj.mean(); e = adj - m; n = len(adj)
    lrv = e @ e / n + 2 * sum((1 - l / (lags + 1)) * (e[l:] @ e[:-l]) / n for l in range(1, lags + 1))
    t_ = m / np.sqrt(lrv / n)
    return float(t_), float(1 - stats.norm.cdf(t_))

def design(D, cols, syms):
    X = [np.ones(len(D))] + [D[c].to_numpy() for c in cols] + [(D["symbol"] == s).to_numpy(float) for s in syms[1:]]
    return np.column_stack(X)

def evaluate(sub, outcome, cand):
    A0 = sub[sub["split"] == "train"].copy(); B0 = sub[sub["split"] == "test"].copy()
    lo, hi = A0[cand].quantile([0.01, 0.99]); A0["cand_w"] = A0[cand].clip(lo, hi); B0["cand_w"] = B0[cand].clip(lo, hi)
    need = CTRL + ["cand_w", outcome]
    A = A0.dropna(subset=need); B = B0.dropna(subset=need).sort_values(["date", "symbol", "hour"])
    if len(A) < 200 or len(B) < 200 or A[outcome].std() < 1e-8 or A["cand_w"].std() < 1e-8:
        return {"skipped": "too few blocks or no variation in outcome/candidate", "n_train": len(A), "n_test": len(B),
                "clark_west_p": np.nan, "delta_r2": np.nan, "delta_r2_ci": [np.nan, np.nan]}
    B = B[B["symbol"].isin(set(A["symbol"]))]
    syms = sorted(A["symbol"].unique()); y = B[outcome].to_numpy()
    bb = np.linalg.lstsq(design(A, CTRL, syms), A[outcome].to_numpy(), rcond=None)[0]
    bf = np.linalg.lstsq(design(A, CTRL + ["cand_w"], syms), A[outcome].to_numpy(), rcond=None)[0]
    fb = design(B, CTRL, syms) @ bb; ff = design(B, CTRL + ["cand_w"], syms) @ bf
    ybar = A[outcome].mean(); sst = np.sum((y - ybar) ** 2)
    t_, p_ = cw_test(y, fb, ff)
    tail = (y > A[outcome].quantile(0.9)).astype(int)
    Rb = rng(88); dd = B["date"].to_numpy(); ud = np.unique(dd)
    pos = {d: np.where(dd == d)[0] for d in ud}; dR2, dA = [], []
    for _ in range(2000):
        idx = np.concatenate([pos[d] for d in Rb.choice(ud, len(ud))])
        s_ = np.sum((y[idx] - ybar) ** 2)
        dR2.append((np.sum((y[idx] - fb[idx]) ** 2) - np.sum((y[idx] - ff[idx]) ** 2)) / s_)
        dA.append(auc_(tail[idx], ff[idx]) - auc_(tail[idx], fb[idx]))
    return {"n_train": len(A), "n_test": len(B), "n_symbols": len(syms),
            "r2_base": 1 - np.sum((y - fb) ** 2) / sst, "r2_full": 1 - np.sum((y - ff) ** 2) / sst,
            "delta_r2": (np.sum((y - fb) ** 2) - np.sum((y - ff) ** 2)) / sst,
            "delta_r2_ci": [float(np.nanquantile(dR2, .025)), float(np.nanquantile(dR2, .975))],
            "auc_base": auc_(tail, fb), "auc_full": auc_(tail, ff),
            "delta_auc_ci": [float(np.nanquantile(dA, .025)), float(np.nanquantile(dA, .975))],
            "clark_west_t": t_, "clark_west_p": p_, "coef_candidate_train": float(bf[len(CTRL) + 1])}

assert os.path.exists(px_path), "run X5 first"
n_unl = unlock_x()
thin = XB[XB["tercile"] == "thin"]
primary = evaluate(thin, "MAE_60", "C3_hat")
secondary = {"thin: MAE_60 | peak_ratio_hat": evaluate(thin, "MAE_60", "peak_ratio_hat"),
             "thin: depth_collapse_60 | C3_hat": evaluate(thin, "depth_collapse_60", "C3_hat"),
             "thin: OI_drop_60 | C3_hat": evaluate(thin, "OI_drop_60", "C3_hat"),
             "mid: MAE_60 | C3_hat": evaluate(XB[XB["tercile"] == "mid"], "MAE_60", "C3_hat"),
             "thick: MAE_60 | C3_hat": evaluate(XB[XB["tercile"] == "thick"], "MAE_60", "C3_hat"),
             "all symbols: MAE_60 | C3_hat": evaluate(XB, "MAE_60", "C3_hat")}
ps = sorted([(v["clark_west_p"] if np.isfinite(v["clark_west_p"]) else 1.0, k) for k, v in secondary.items()])
for i, (p_, k) in enumerate(ps):
    secondary[k]["holm_p"] = min(1.0, max(p2 * (len(ps) - j) for j, (p2, _) in enumerate(ps[:i + 1])))
print(json.dumps(_clean(primary), indent=1))
print(pd.DataFrame(secondary).T[["delta_r2", "delta_r2_ci", "clark_west_p", "holm_p"]].to_string())
gx = bool(np.isfinite(primary["clark_west_p"]) and primary["clark_west_p"] < 0.05 and primary["delta_r2_ci"][0] > 0)
record("X6_xsec_test_G2", bool(gx), "real-data",
       {"primary": primary, "secondary": secondary, "prereg_sha256": X_HASH, "test_set_unlocks": n_unl},
       criterion=PREREG_X["success"], note="Report whatever it shows; unlocks > 1 must be disclosed.")

# flags in the thin tercile: high C3 vs high volatility (each flags 10% of training blocks)
A = thin[thin["split"] == "train"].dropna(subset=["C3_hat", "rv_60", "MAE_60"])
B = thin[thin["split"] == "test"].dropna(subset=["C3_hat", "rv_60", "MAE_60"])
tail = (B["MAE_60"] > A["MAE_60"].quantile(0.9)).to_numpy()
def pr_(f):
    tp = np.sum(f & tail); return {"flag_rate": f.mean(), "precision": tp / max(f.sum(), 1), "recall": tp / max(tail.sum(), 1)}
fw = (B["C3_hat"] > A["C3_hat"].quantile(0.9)).to_numpy(); fv = (B["rv_60"] > A["rv_60"].quantile(0.9)).to_numpy()
record("X6b_thin_flags", None, "real-data", {"C3_flag": pr_(fw), "vol_flag": pr_(fv), "either": pr_(fw | fv),
       "n_tail_events": int(tail.sum())}, note="Thin books: does the C3 flag catch tail hours a volatility flag misses?")


## Exploratory: natural experiment and trade-level depletion

In [ ]:
# ============================================================================
# CELL X7 — NATURAL EXPERIMENT: TICK-SIZE CHANGES (exploratory, events found from the data)
# The exchange occasionally changes a contract's price tick. A coarser tick changes queue
# lengths and hence how fast depth refills (r). We do not hard-code dates: the tick is inferred
# every day from the price grid (X2), changes are located on the panel, then dated exactly by
# a binary search over daily kline files.
# Design: 10 days before vs 10 days after (event day dropped), treated symbol vs up to 4 control
# symbols of similar liquidity whose tick did not change. Difference-in-differences, day bootstrap.
# T6 link: refill r governs the height and duration of any overshoot; the amplification condition
# C3 > 0 depends on φ, ε, κ, a, η, q, which a tick change may also move.
# ============================================================================
EVENT_WIN, MAX_EVENTS, N_CTRL = 10, 12, 4
TK = vst.dropna(subset=["tick"])[["symbol", "date", "tick"]].sort_values(["symbol", "date"])

def daily_tick(s, d):
    raw = fetch("klines", s, d)
    return np.nan if raw is None else infer_tick(read_zip_csv(raw, KL_COLS))

events = []
for s, g in TK.groupby("symbol"):
    v, dts = g["tick"].to_numpy(), g["date"].tolist()
    for i in range(2, len(v) - 1):
        ratio = v[i] / v[i - 1]
        if (ratio >= 2 or ratio <= 0.5) and v[i - 2] == v[i - 1] and v[i + 1] == v[i]:
            days = pd.date_range(dts[i - 1], dts[i], freq="D").strftime("%Y-%m-%d").tolist()
            lo, hi = 0, len(days) - 1            # first day carrying the new tick
            while lo < hi:
                m = (lo + hi) // 2
                tm = daily_tick(s, days[m])
                if np.isfinite(tm) and np.isclose(tm, v[i]):
                    hi = m
                else:
                    lo = m + 1
            events.append({"symbol": s, "event_date": days[lo], "tick_before": float(v[i - 1]),
                           "tick_after": float(v[i]), "ratio": float(ratio)})
events = sorted(events, key=lambda e: e["event_date"])[:MAX_EVENTS]
print(f"tick changes found: {len(events)}")
for e in events:
    print("  ", e)

liq_rank = {s: i for i, s in enumerate(sym.index)}         # X3 ranking (deep -> thin)
def stable_tick(s, d0, days=60):
    g = TK[(TK["symbol"] == s) & (TK["date"] >= (pd.Timestamp(d0) - pd.Timedelta(days=days)).strftime("%Y-%m-%d")) &
           (TK["date"] <= (pd.Timestamp(d0) + pd.Timedelta(days=days)).strftime("%Y-%m-%d"))]["tick"]
    return len(g) >= 3 and g.nunique() == 1

def daily_stats(s, d):
    df, st = get_day(s, d)
    if df is None or not st.get("depth_day_ok"):
        return None
    p, u, dd_, x, tot = prep(df, FLOW_SCALE.get(s, 1.0))
    est = fit_t6(p, u, dd_, x)
    if est is None:
        return None
    return {"r": est["r"], "eps": est["eps"], "phi": est["phi"], "C3": est["C3"], "peak_ratio": est["peak_ratio"],
            "log_depth1": float(np.log(np.nanmedian(tot))), "rv": float(np.nanstd(np.diff(p))),
            "rel_tick_bp": float(st["tick"] / np.nanmedian(df["close"]) * 1e4) if np.isfinite(st.get("tick", np.nan)) else np.nan}

OUTS = ["r", "eps", "phi", "C3", "peak_ratio", "log_depth1", "rv", "rel_tick_bp"]
res7 = []
for e in events:
    s, d0 = e["symbol"], e["event_date"]
    ctrls = sorted([c for c in SYMBOLS if c != s and c in liq_rank and stable_tick(c, d0)],
                   key=lambda c: abs(liq_rank[c] - liq_rank.get(s, 0)))[:N_CTRL]
    rel = [k for k in range(-EVENT_WIN, EVENT_WIN + 1) if k != 0]
    dates = {k: (pd.Timestamp(d0) + pd.Timedelta(days=k)).strftime("%Y-%m-%d") for k in rel}
    panel_e = []
    with ThreadPoolExecutor(THREADS) as ex:
        futs = {ex.submit(daily_stats, c, dates[k]): (c, k) for c in [s] + ctrls for k in rel}
        for f in as_completed(futs):
            c, k = futs[f]
            try:
                r_ = f.result()
            except Exception:
                r_ = None
            if r_:
                panel_e.append({**r_, "symbol": c, "k": k, "treated": c == s, "post": k > 0})
    P = pd.DataFrame(panel_e)
    if P.empty or P["treated"].sum() < 8 or (~P["treated"]).sum() < 8:
        res7.append({**e, "status": "insufficient valid days"}); continue
    Rb = rng(70); did, ci = {}, {}
    def did_of(Q):
        out = {}
        for o in OUTS:
            m = Q.groupby(["treated", "post"])[o].mean()
            try:
                out[o] = (m[(True, True)] - m[(True, False)]) - (m[(False, True)] - m[(False, False)])
            except KeyError:
                out[o] = np.nan
        return out
    did = did_of(P)
    ks = P["k"].unique(); boots = []
    for _ in range(1000):
        pick = Rb.choice(ks, len(ks)); boots.append(did_of(pd.concat([P[P["k"] == k] for k in pick])))
    B = pd.DataFrame(boots)
    ci = {o: [float(B[o].quantile(.025)), float(B[o].quantile(.975))] for o in OUTS}
    res7.append({**e, "controls": ctrls, "n_obs": len(P), "did": did, "did_ci": ci})
    print(s, d0, {o: round(did[o], 4) for o in ["rel_tick_bp", "r", "eps", "C3", "peak_ratio"]})

ok7 = [x for x in res7 if "did" in x]
summary7 = {}
if len(ok7) >= 3:
    dr = np.array([x["did"]["r"] for x in ok7]); dpk = np.array([x["did"]["peak_ratio"] for x in ok7])
    summary7 = {"n_events": len(ok7),
                "spearman_dr_vs_dpeak": float(stats.spearmanr(dr, dpk).statistic),
                "share_events_r_up_after_coarser_tick": float(np.mean([x["did"]["r"] > 0 for x in ok7 if x["ratio"] > 1])) if any(x["ratio"] > 1 for x in ok7) else None}
record("X7_tick_natural_experiment", None, "real-data", {"events": res7, "summary": summary7},
       note="Exploratory: events are found from the data, not pre-registered. With few events, report each one. "
            "T6 predicts that a rise in r lowers any overshoot (negative Spearman of Δr vs Δpeak).")


In [ ]:
# ============================================================================
# CELL X8 — DEPLETION AT THE SNAPSHOT SCALE (~30 s) FROM TRADE DATA
# Part 2 found ε < 0 at one minute: taker buying is followed by slightly MORE ask-side depth,
# i.e. the book refills the side being hit. Is that a 1-minute aggregation artifact?
# Here signed taker flow is summed exactly between consecutive validated depth snapshots
# using aggTrades (every trade, with the aggressor side), and we estimate
#     d_i = c + (1-r) d_{i-1} + ε30 f_i + e           (depth imbalance within 1%, unit-free)
# with flow in units of the median |f| per interval. Sweeps: ε30 for intervals in the top decile of |f|.
# T6's depletion channel needs ε30 > 0 (flow takes depth from the side it hits).
# BTC aggTrades files are large (100s of MB per day): AT_DAYS keeps this manageable.
# ============================================================================
AT_DAYS = 6
AGG_COLS = ["agg_trade_id", "price", "quantity", "first_trade_id", "last_trade_id", "transact_time", "is_buyer_maker"]
thin_syms = [s for s in sym.index if TERCILE[s] == "thin"][-3:]
mid_syms = [s for s in sym.index if TERCILE[s] == "mid"][:1]
AT_SYMS = [s for s in ["BTCUSDT"] if s in TERCILE] + mid_syms + thin_syms

def read_agg(raw):
    """Memory-light reader for large aggTrades files: price, quantity, time, aggressor side only."""
    with zipfile.ZipFile(io.BytesIO(raw)) as z:
        name = [n for n in z.namelist() if n.endswith(".csv")][0]
        with z.open(name) as fh:
            first = fh.readline().decode()
        has_header = bool(re.search(r"[A-Za-z_]", first.split(",")[0]))
        with z.open(name) as fh:
            a = pd.read_csv(fh, header=0 if has_header else None, usecols=[1, 2, 5, 6],
                            dtype={1: "float64", 2: "float64"} if not has_header else None)
    a.columns = ["price", "quantity", "transact_time", "is_buyer_maker"]
    return a

def interval_series(s, d):
    cache = os.path.join(XDATA, "depl30", f"{s}_{d}.parquet"); os.makedirs(os.path.dirname(cache), exist_ok=True)
    if os.path.exists(cache):
        return pd.read_parquet(cache)
    t0 = int(pd.Timestamp(f"{d} 00:00:00", tz="UTC").timestamp())
    snap, _ = validated_snapshots(s, d, PANEL[(s, d)]["close"].to_numpy(), t0)
    raw = fetch("aggTrades", s, d)
    if snap is None or raw is None or len(snap) < 500:
        return None
    a = read_agg(raw); del raw
    sec = to_utc_seconds(a["transact_time"])
    bm = a["is_buyer_maker"].astype(str).str.lower().isin(["true", "1"]).to_numpy()
    sgn = np.where(bm, -1.0, 1.0) * a["price"].astype(float).to_numpy() * a["quantity"].astype(float).to_numpy() / 1e6
    snap = snap.sort_index(); ss = snap.index.to_numpy()
    pos = np.searchsorted(ss, sec, side="left")              # trade at second t belongs to interval ending at the next snapshot
    flow = np.bincount(pos[pos < len(ss)], weights=sgn[pos < len(ss)], minlength=len(ss))
    out = pd.DataFrame({"sec": ss, "d": ((snap["bid1"] - snap["ask1"]) / (snap["bid1"] + snap["ask1"])).to_numpy(),
                        "flow": flow, "gap": np.diff(ss, prepend=ss[0] - 30)})
    out.to_parquet(cache)
    return out

res8 = {}
for s in AT_SYMS:
    days = [d for (s2, d) in OK if s2 == s and d <= TRAIN_END]
    days = [days[int(i)] for i in np.linspace(0, len(days) - 1, min(AT_DAYS, len(days)))] if days else []
    per = []
    for d in days:
        try:
            iv = interval_series(s, d)
        except Exception as e:
            print("  !", s, d, e); iv = None
        if iv is None:
            continue
        iv = iv[(iv["gap"] > 0) & (iv["gap"] <= 90)]           # consecutive snapshots only
        sc = np.nanmedian(np.abs(iv["flow"])) or 1.0
        f = iv["flow"].to_numpy() / sc; dd = iv["d"].to_numpy()
        big = np.abs(f) >= np.quantile(np.abs(f), 0.9)
        y, X = dd[1:], np.column_stack([dd[:-1], f[1:], f[1:] * big[1:], f[:-1]])
        ok = np.all(np.isfinite(X), 1) & np.isfinite(y)
        b = np.linalg.lstsq(np.column_stack([np.ones(ok.sum()), X[ok]]), y[ok], rcond=None)[0]
        p_, u_, d1, x_, _ = prep(PANEL[(s, d)], FLOW_SCALE[s]); e1 = fit_t6(p_, u_, d1, x_)
        per.append({"date": d, "eps30": b[2], "eps30_sweep_extra": b[3], "eps30_next": b[4], "rho": b[1],
                    "eps_1min": e1["eps"] if e1 else np.nan, "n_intervals": int(ok.sum())})
    if per:
        P = pd.DataFrame(per); R8 = rng(80)
        res8[s] = {"tercile": TERCILE[s], "days": P.to_dict("records"),
                   **{f"mean_{c}": float(P[c].mean()) for c in ["eps30", "eps30_sweep_extra", "eps30_next", "eps_1min"]},
                   **{f"ci_{c}": boot_ci(P[c], R8) for c in ["eps30", "eps30_sweep_extra", "eps30_next"]},
                   "share_days_eps30_pos": float((P["eps30"] > 0).mean())}
        print(f"{s:10s} ({TERCILE[s]}): ε30 {res8[s]['mean_eps30']:+.5f} CI {np.round(res8[s]['ci_eps30'], 5)}; "
              f"sweep extra {res8[s]['mean_eps30_sweep_extra']:+.5f}; 1-min ε {res8[s]['mean_eps_1min']:+.5f}")
record("X8_depletion_30s", None, "real-data", {"by_symbol": res8, "days_per_symbol": AT_DAYS},
       note="T6's depletion channel requires ε30 > 0 (CI above 0), at least for sweeps. If ε30 < 0 too, "
            "the 1-minute finding is not an aggregation artifact: the book refills the side being hit.")


## Paper 1: disclosure test

In [ ]:
# ============================================================================
# CELL X9 — DISCLOSURE TEST FOR PAPER 1 (class-level positions in Binance `metrics`)
# Binance publishes, every 5 minutes, long/short ratios for its "top trader" group (by position
# and by account) and for all accounts. These are class-tagged position data, the kind of
# disclosure Paper 1's Proposition (tagged flow) is about.
#   Tagged model, class c:  n_c,t = r_c n_c,t-1 + w_c ret_t + e,   n = (R-1)/(R+1) from the L/S ratio R.
#   Price-only model:       f_t = Σ_{m=1..12} k_m ret_{t-m} + e    (5-min signed taker flow),
#                           then k_m = w1 r1^(m-1) + w2 r2^(m-1)   (two memory rates from prices and flow only).
# Day-block bootstrap (training days). Paper 1 predicts: rates are pinned down tightly with tagged
# data and loosely without it; the ratio of CI widths measures what disclosure buys.
# ============================================================================
DISC_SYMS = [s for s in ["BTCUSDT", "ETHUSDT"] if s in TERCILE] + [s for s in sym.index if TERCILE[s] == "thin"][-2:]
CLASSES = {"top_position": "top_pos_ratio", "top_account": "top_acc_ratio", "all_accounts": "acc_ratio"}

def five_min(df, s):
    m = df.index[df["top_pos_ratio"].notna() | df["acc_ratio"].notna()].to_numpy()
    m = m[m >= 5]
    if len(m) < 100:
        return None
    p = np.log(df["close"].to_numpy()) * 1e4; fl = df["flow"].to_numpy() / FLOW_SCALE[s]
    cf = np.concatenate([[0.0], np.cumsum(fl)])
    out = pd.DataFrame({"m": m, "ret": p[m] - p[m - 5], "flow": cf[m + 1] - cf[m - 4]})
    for c, col in CLASSES.items():
        R = df[col].to_numpy()[m]
        out[c] = (R - 1) / (R + 1)
    return out

def tagged_fit(days5, c):
    Y, X = [], []
    for g in days5:
        n = g[c].to_numpy(); r = g["ret"].to_numpy()
        y, x1, x2 = n[1:], n[:-1], r[1:]
        ok = np.isfinite(y) & np.isfinite(x1) & np.isfinite(x2)
        if ok.sum() < 30:
            continue
        Y.append(y[ok] - y[ok].mean()); X.append(np.column_stack([x1[ok] - x1[ok].mean(), x2[ok] - x2[ok].mean()]))
    if not Y:
        return np.nan, np.nan
    b = np.linalg.lstsq(np.vstack(X), np.concatenate(Y), rcond=None)[0]
    return float(b[0]), float(b[1])

def kernel_fit(days5, L=12):
    Y, X = [], []
    for g in days5:
        f, r = g["flow"].to_numpy(), g["ret"].to_numpy()
        if len(f) <= L + 10:
            continue
        Xl = np.column_stack([r[L - m:len(r) - m] for m in range(1, L + 1)]); y = f[L:]
        ok = np.all(np.isfinite(Xl), 1) & np.isfinite(y)
        Y.append(y[ok] - y[ok].mean()); X.append(Xl[ok] - Xl[ok].mean(0))
    k = np.linalg.lstsq(np.vstack(X), np.concatenate(Y), rcond=None)[0]
    m = np.arange(L)
    def resid(th):
        w1, r1, w2, r2 = th
        return w1 * r1 ** m + w2 * r2 ** m - k
    best = None
    for r1, r2 in [(0.2, 0.8), (0.1, 0.6), (0.4, 0.95), (0.05, 0.3)]:
        try:
            sol = optimize.least_squares(resid, [k[0] / 2, r1, k[0] / 2, r2],
                                         bounds=([-np.inf, 0, -np.inf, 0], [np.inf, 0.99, np.inf, 0.99]))
            if best is None or sol.cost < best.cost:
                best = sol
        except Exception:
            pass
    w1, r1, w2, r2 = best.x
    if r1 > r2:
        w1, r1, w2, r2 = w2, r2, w1, r1
    return {"k": k, "r_fast": r1, "r_slow": r2, "w_fast": w1, "w_slow": w2}

res9 = {}
for s in DISC_SYMS:
    days5 = [g for g in (five_min(PANEL[(s2, d)], s) for (s2, d) in OK if s2 == s and d <= TRAIN_END) if g is not None]
    if len(days5) < 15:
        print(s, "too few days with metrics"); continue
    R9 = rng(90); n = len(days5)
    point = {c: tagged_fit(days5, c) for c in CLASSES}; kp = kernel_fit(days5)
    bt = {c: [] for c in CLASSES}; bk = []
    for _ in range(200):
        pick = [days5[i] for i in R9.integers(0, n, n)]
        for c in CLASSES:
            bt[c].append(tagged_fit(pick, c))
        try:
            kk = kernel_fit(pick); bk.append((kk["r_fast"], kk["r_slow"]))
        except Exception:
            pass
    q = lambda v: [float(np.nanquantile(v, .025)), float(np.nanquantile(v, .975))]
    tag = {c: {"r": point[c][0], "w": point[c][1], "r_ci": q([b[0] for b in bt[c]]), "w_ci": q([b[1] for b in bt[c]])} for c in CLASSES}
    diff = np.array([b[0] for b in bt["top_position"]]) - np.array([b[0] for b in bt["all_accounts"]])
    bk = np.array(bk)
    pr = {"r_fast": kp["r_fast"], "r_slow": kp["r_slow"], "r_fast_ci": q(bk[:, 0]), "r_slow_ci": q(bk[:, 1]), "kernel": kp["k"]}
    w_tag = np.mean([tag[c]["r_ci"][1] - tag[c]["r_ci"][0] for c in CLASSES])
    w_pr = np.mean([pr["r_fast_ci"][1] - pr["r_fast_ci"][0], pr["r_slow_ci"][1] - pr["r_slow_ci"][0]])
    res9[s] = {"n_days": n, "tagged": tag, "top_minus_all_rate_ci": q(diff), "price_only": pr,
               "mean_rate_ci_width_tagged": float(w_tag), "mean_rate_ci_width_price_only": float(w_pr),
               "width_ratio_price_only_over_tagged": float(w_pr / w_tag) if w_tag > 0 else np.nan}
    print(f"{s}: rate CI width tagged {w_tag:.4f} vs price-only {w_pr:.4f} (ratio {w_pr / w_tag:.1f}); "
          f"top-minus-all rate CI {np.round(q(diff), 4)}")
record("X9_disclosure_metrics", None, "real-data", {"by_symbol": res9},
       note="Paper 1 predicts a width ratio well above 1: class-tagged data pin down memory rates that "
            "price and flow data alone leave loose. The top-minus-all CI shows whether classes differ.")


## Summary

In [ ]:
# ============================================================================
# CELL X10 — SUMMARY AND DOWNLOAD (Part 4)
# ============================================================================
rows = []
for fp in sorted(glob.glob(os.path.join(RESULTS, "X*.json"))):
    r = json.load(open(fp)); rows.append({"test": r["test_id"], "status": r["status"], "kind": r["kind"]})
print(pd.DataFrame(rows).to_string(index=False))
zp = os.path.join(BASE, "xsec_results.zip")
with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
    for fp in glob.glob(os.path.join(RESULTS, "X*.json")) + [os.path.join(RESULTS, "xsec_manifest_sha256.csv")] + \
              glob.glob(os.path.join(PREREG, "P4_*.json")) + glob.glob(os.path.join(PREREG, "test_set_unlocks_xsec.json")):
        if os.path.exists(fp):
            z.write(fp, os.path.relpath(fp, BASE))
print("zipped ->", zp)
if IN_COLAB:
    from google.colab import files
    files.download(zp)


## How to read the results
| Test | Supports the theory if | If not |
|---|---|---|
| X3 | Spearman(log depth, mean $C_3$) < 0 with CI below 0; thin tercile has more days with $C_3>0$ | liquidity does not move books toward amplification |
| X4 | thin tercile: overshoot (B) CI > 0 and much smaller in (C) | no depth-carried overshoot even in thin books |
| **X6** | Clark–West p < 0.05 **and** ΔR² CI > 0 in the thin tercile | report the null; T6 does not forecast where books are thin |
| X7 | events where Δr and Δpeak move in opposite directions | report each event; few events are descriptive only |
| X8 | ε30 > 0 (CI above 0), at least for sweeps | the replenishment result is real, not an aggregation artefact |
| X9 | CI-width ratio (price-only / tagged) well above 1 | disclosure adds little in this market |

Send me `xsec_results.zip` when it finishes and I will put the results into both papers.